In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
from google.colab import files

print("scikit-learn version:", sklearn.__version__)   # note this down for later

scikit-learn version: 1.6.1


In [2]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

try:
    df = pd.read_csv(filename)
except UnicodeDecodeError:
    df = pd.read_csv(filename, encoding='utf-16')   # some Kaggle SQLi files use this encoding

print(df.shape)
df.head()

Saving Modified_SQL_Dataset.csv.zip to Modified_SQL_Dataset.csv.zip
(30919, 2)


,Query,Label
0,""" or pg_sleep ( __TIME__ ) --",1
1,create user name identified by pass123 tempora...,1
2,AND 1 = utl_inaddr.get_host_address ( ...,1
3,select * from users where id = '1' or @ @1 ...,1
4,"select * from users where id = 1 or 1#"" ( ...",1


In [3]:
df = df.iloc[:, :2]     # keep only the first two columns (the query and the label)

label_col = [c for c in df.columns if df[c].nunique() <= 3][0]
query_col = [c for c in df.columns if c != label_col][0]
df = df[[query_col, label_col]]
df.columns = ['query', 'label']

df = df.dropna()
df['query'] = df['query'].astype(str)
df['label'] = df['label'].astype(int)

before = len(df)
df = df.drop_duplicates()
print("rows before/after removing duplicates:", before, "->", len(df))
print(df['label'].value_counts())

rows before/after removing duplicates: 30919 -> 30907
label
0    19529
1    11378
Name: count, dtype: int64


In [4]:
X = df['query']      # the information we use (the query text)
y = df['label']      # the answer we want to predict (0 = benign, 1 = malicious)

train_X, val_X, train_y, val_y = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("training rows:", len(train_X), "  validation rows:", len(val_X))
print("malicious share in training:", round(train_y.mean(), 3))
print("malicious share in validation:", round(val_y.mean(), 3))

training rows: 24725   validation rows: 6182
malicious share in training: 0.368
malicious share in validation: 0.368


In [5]:
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=20000)

train_vec = vectorizer.fit_transform(train_X)   # learn the patterns from TRAINING queries only
val_vec = vectorizer.transform(val_X)           # apply those same patterns to the validation queries

print("training matrix shape:", train_vec.shape)
print("validation matrix shape:", val_vec.shape)

training matrix shape: (24725, 20000)
validation matrix shape: (6182, 20000)


In [6]:
model = LogisticRegression(max_iter=1000)    # create the empty model
model.fit(train_vec, train_y)                # STUDY: learn from the training table + answers

preds = model.predict(val_vec)               # GUESS: classify the held-back queries

print(classification_report(val_y, preds, target_names=['benign', 'malicious']))
print(confusion_matrix(val_y, preds))

              precision    recall  f1-score   support

      benign       0.99      1.00      1.00      3906
   malicious       1.00      0.99      0.99      2276

    accuracy                           1.00      6182
   macro avg       1.00      0.99      0.99      6182
weighted avg       1.00      1.00      1.00      6182

[[3905    1]
 [  28 2248]]


In [7]:
rf = RandomForestClassifier(n_estimators=100, random_state=1, n_jobs=-1)
rf.fit(train_vec, train_y)

rf_preds = rf.predict(val_vec)

print(classification_report(val_y, rf_preds, target_names=['benign', 'malicious']))
print(confusion_matrix(val_y, rf_preds))

              precision    recall  f1-score   support

      benign       1.00      1.00      1.00      3906
   malicious       1.00      0.99      0.99      2276

    accuracy                           1.00      6182
   macro avg       1.00      1.00      1.00      6182
weighted avg       1.00      1.00      1.00      6182

[[3896   10]
 [  13 2263]]


In [8]:
names = np.array(vectorizer.get_feature_names_out())   # the 20,000 chunk names
order = np.argsort(model.coef_[0])                     # chunks sorted from most benign to most malicious

print("Patterns pointing to BENIGN:")
print(list(names[order[:15]]))
print()
print("Patterns pointing to MALICIOUS:")
print(list(names[order[-15:]]))

Patterns pointing to BENIGN:
[', ', 'ro', ' f', 'm ', ' ) ;', ') ;', 'fr', 'om', 'ar', ' ;', 'rom', 'om ', 'ra', 'fro', 'from']

Patterns pointing to MALICIOUS:
['ll s', ',6', "1'", '" ', ',8', ',4', '1 ', "' ", '--', ' = ', '= ', ' =', ' )', ') ', ' ) ']


In [9]:
missed = val_X[(val_y == 1) & (preds == 0)]     # real attacks that the model called benign
print(len(missed), "attacks were missed. Some examples:\n")
for q in missed.head(10):
    print("-", q[:150])

28 attacks were missed. Some examples:

- insert
- c
- @variable
- admin" #
- //
- ; execute immediate 'sel' || 'ect us' || 'er'
- asc
- desc
- or 'unusual'   =   'unusual'
- ?s


In [10]:
tests = [
    # normal inputs (should be low)
    "SELECT name FROM users WHERE id = 5",
    "hello how are you",
    "My name is O'Brien",                        # an apostrophe, but harmless
    "SELECT * FROM orders WHERE status = 'shipped'",
    # classic attacks (should be high)
    "' OR '1'='1' --",
    "admin' --",
    "1; DROP TABLE users",
    "1 UNION SELECT username, password FROM users",
    # trickier attacks
    "' OR 1=1 #",
    "' oR '1'='1",                               # mixed capitals
    "%27%20OR%201%3D1--",                        # the same attack, URL-encoded
    "'; EXEC('sel' + 'ect * from users') --",    # keyword split into pieces
]

probs = model.predict_proba(vectorizer.transform(tests))[:, 1]
for t, p in zip(tests, probs):
    flag = "MALICIOUS" if p >= 0.5 else "benign   "
    print(f"{p:.2f}  {flag}  {t}")

0.50  MALICIOUS  SELECT name FROM users WHERE id = 5
0.03  benign     hello how are you
0.05  benign     My name is O'Brien
0.02  benign     SELECT * FROM orders WHERE status = 'shipped'
0.95  MALICIOUS  ' OR '1'='1' --
0.67  MALICIOUS  admin' --
0.37  benign     1; DROP TABLE users
0.44  benign     1 UNION SELECT username, password FROM users
0.96  MALICIOUS  ' OR 1=1 #
0.89  MALICIOUS  ' oR '1'='1
0.45  benign     %27%20OR%201%3D1--
0.26  benign     '; EXEC('sel' + 'ect * from users') --


In [11]:
joblib.dump(model, 'sqli_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

files.download('sqli_model.pkl')
files.download('vectorizer.pkl')

print("scikit-learn version used for training:", sklearn.__version__)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

scikit-learn version used for training: 1.6.1
